# Forecast-Driven Carbon-Aware Edge-Cloud Task Offloading
## End-to-end ML notebook for the cloud research project

**Project objective:** forecast task execution duration and regional carbon intensity, then use those predictions in a simple carbon-aware edge/cloud scheduler.

**Selected first model:** `RandomForestRegressor` from scikit-learn. It is a practical nonlinear baseline and is easier to integrate with the planned Java CloudSim/EdgeCloudSim simulation than a deep-learning model.

### What this notebook does
1. Loads and inspects `master_dataset(1).xlsx`.
2. Cleans and prepares task-level data.
3. Trains a duration regression model and evaluates it on a chronological holdout.
4. Builds regional carbon-intensity time series and trains a one-step-ahead forecasting model with lag features.
5. Evaluates both models using MAE, RMSE and R².
6. Uses the forecasts in an illustrative edge/cloud scheduling policy.
7. Compares decisions and estimated energy/carbon under the stated assumptions.
8. Saves trained models and prediction outputs for later integration.

> **Scientific honesty:** this dataset does not contain a direct server CPU-utilization label. This notebook predicts task duration and carbon intensity. Server utilization and energy are estimated using explicit assumptions, not presented as measured values. The scheduler is a research prototype, not a production deployment.

## 0. Install/import dependencies

If an import fails in your Jupyter environment, run the install command below in a separate cell.

In [ ]:
# Uncomment if required:
# %pip install pandas numpy scikit-learn matplotlib seaborn openpyxl joblib

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

## 1. Load the Excel dataset

Place this notebook in the same folder as `master_dataset(1).xlsx`, or update `DATA_PATH` to the file's location.

In [ ]:
DATA_PATH = Path("master_dataset(1).xlsx")

if not DATA_PATH.exists():
    # Also check the current working directory for any xlsx file as a convenience.
    candidates = list(Path.cwd().glob("*.xlsx"))
    if len(candidates) == 1:
        DATA_PATH = candidates[0]
    else:
        raise FileNotFoundError(
            "Could not find master_dataset(1).xlsx. Put the file beside this notebook "
            "or update DATA_PATH to its full path."
        )

df = pd.read_excel(DATA_PATH)
print("Dataset path:", DATA_PATH.resolve())
print("Shape:", df.shape)
display(df.head())

In [ ]:
print("Columns:", len(df.columns))
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "unique_values": df.nunique()
}).sort_values(["missing_values", "unique_values"], ascending=[False, True]))

print("Duplicate rows:", df.duplicated().sum())
if "task_id" in df.columns:
    print("Duplicate task IDs:", df["task_id"].duplicated().sum())

## 2. Data quality checks and basic exploration

We keep the original task order/timestamps available for chronological splitting. We do not randomly split the rows because that can make future observations leak into training.

In [ ]:
# Parse timestamps safely and sort chronologically.
for col in ["sim_timestamp_utc", "carbon_timestamp_utc"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

df = df.drop_duplicates(subset=["task_id"]).copy() if "task_id" in df.columns else df.copy()
df = df.sort_values("sim_offset_sec" if "sim_offset_sec" in df.columns else "arrival_time_trace_sec").reset_index(drop=True)

numeric_summary_cols = [
    c for c in [
        "task_size_mb", "cpu_demand", "gpu_demand", "observed_duration_sec",
        "delay_tolerance_sec", "carbon_intensity_local", "carbon_intensity_cloud"
    ] if c in df.columns
]
display(df[numeric_summary_cols].describe().T)

print("Priority distribution:")
display(df["priority"].value_counts(dropna=False).to_frame("tasks") if "priority" in df.columns else "No priority column")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df["observed_duration_sec"].clip(upper=df["observed_duration_sec"].quantile(0.99)).hist(bins=40, ax=axes[0])
axes[0].set_title("Task duration (clipped at 99th percentile)")
axes[0].set_xlabel("Seconds")
df["task_size_mb"].hist(bins=35, ax=axes[1])
axes[1].set_title("Task size")
axes[1].set_xlabel("MB")
df["carbon_intensity_local"].hist(bins=35, alpha=0.7, label="Local", ax=axes[2])
df["carbon_intensity_cloud"].hist(bins=35, alpha=0.7, label="Cloud", ax=axes[2])
axes[2].set_title("Carbon intensity")
axes[2].set_xlabel("Dataset units (typically gCO₂/kWh)")
axes[2].legend()
plt.tight_layout()
plt.show()

## 3. Model A — predict task execution duration

### Target
`observed_duration_sec`

### Features
Known task properties such as task size, CPU/GPU demand, priority, device type, device capacity, and region. We intentionally exclude IDs and observed outcomes that could leak the target.

We use an 80/20 **chronological split**. The first 80% trains the model; the latest 20% is held out for evaluation.

In [ ]:
target_duration = "observed_duration_sec"

duration_numeric_candidates = [
    "task_size_mb", "cpu_demand", "gpu_demand", "device_id",
    "local_processing_rate_mhz", "device_cores", "battery_capacity_mah",
    "transmit_power_dbm", "device_location_x_m", "device_location_y_m",
    "delay_tolerance_sec"
]
duration_categorical_candidates = ["priority", "device_type", "device_region", "radio_type"]

duration_numeric = [c for c in duration_numeric_candidates if c in df.columns]
duration_categorical = [c for c in duration_categorical_candidates if c in df.columns]
duration_features = duration_numeric + duration_categorical

if not duration_features:
    raise ValueError("No usable duration-model features were found.")

# Keep only rows with a valid target.
duration_df = df.dropna(subset=[target_duration]).copy()
duration_df = duration_df[duration_df[target_duration] >= 0].copy()

split_idx = int(len(duration_df) * 0.80)
train_duration = duration_df.iloc[:split_idx].copy()
test_duration = duration_df.iloc[split_idx:].copy()

print("Features:", duration_features)
print("Train rows:", len(train_duration), "| Test rows:", len(test_duration))
print("Train time range:",
      train_duration["sim_timestamp_utc"].min() if "sim_timestamp_utc" in train_duration else "n/a",
      "to",
      train_duration["sim_timestamp_utc"].max() if "sim_timestamp_utc" in train_duration else "n/a")
print("Test time range:",
      test_duration["sim_timestamp_utc"].min() if "sim_timestamp_utc" in test_duration else "n/a",
      "to",
      test_duration["sim_timestamp_utc"].max() if "sim_timestamp_utc" in test_duration else "n/a")

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

duration_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, duration_numeric),
        ("cat", categorical_transformer, duration_categorical)
    ],
    remainder="drop"
)

duration_model = Pipeline(steps=[
    ("preprocessor", duration_preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=120,
        max_depth=18,
        min_samples_leaf=2,
        max_features=0.9,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

X_train_duration = train_duration[duration_features]
y_train_duration = train_duration[target_duration]
X_test_duration = test_duration[duration_features]
y_test_duration = test_duration[target_duration]

duration_model.fit(X_train_duration, y_train_duration)
pred_duration = np.maximum(0, duration_model.predict(X_test_duration))

In [ ]:
def regression_metrics(y_true, y_pred, label):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    return {
        "model": label,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse,
        "R2": r2_score(y_true, y_pred)
    }

duration_results = pd.DataFrame([
    regression_metrics(y_test_duration, pred_duration, "Random Forest"),
    regression_metrics(
        y_test_duration,
        np.repeat(y_train_duration.median(), len(y_test_duration)),
        "Median baseline"
    )
])
display(duration_results.round(3))

In [ ]:
duration_predictions = test_duration[["task_id", "observed_duration_sec"]].copy()
duration_predictions["predicted_duration_sec"] = pred_duration
duration_predictions["absolute_error_sec"] = (
    duration_predictions["observed_duration_sec"] - duration_predictions["predicted_duration_sec"]
).abs()
display(duration_predictions.head(10))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_test_duration, pred_duration, alpha=0.35)
axes[0].set_xlabel("Actual duration (seconds)")
axes[0].set_ylabel("Predicted duration (seconds)")
axes[0].set_title("Actual vs predicted duration")
axes[1].hist(duration_predictions["absolute_error_sec"].clip(
    upper=duration_predictions["absolute_error_sec"].quantile(0.99)), bins=35)
axes[1].set_title("Absolute prediction error (99% clipped)")
axes[1].set_xlabel("Seconds")
plt.tight_layout()
plt.show()

## 4. Model B — forecast regional carbon intensity one time step ahead

The workbook includes carbon-intensity columns for several regions. We turn these into a long time series, one row per timestamp and region, then construct lag features from **past values only**.

- Target: next observed carbon intensity for a region.
- Features: region, hour/day features, and lagged carbon intensity values.
- Split: chronological, by timestamp.
- Important: this is a one-step-ahead forecast at the dataset's observed time resolution. It should not be described as a multi-hour forecast without additional recursive forecasting and evaluation.

In [ ]:
regions = [
    "california", "desert_southwest", "florida", "mid_atlantic",
    "midwest", "new_england", "new_york", "pacific_northwest", "texas"
]
carbon_columns = [f"carbon_intensity_{r}" for r in regions if f"carbon_intensity_{r}" in df.columns]
if "carbon_timestamp_utc" not in df.columns or not carbon_columns:
    raise ValueError("Carbon forecast requires carbon_timestamp_utc and regional carbon-intensity columns.")

# Duplicate task rows can share a carbon timestamp. Average duplicates at timestamp-region level.
carbon_wide = (
    df.dropna(subset=["carbon_timestamp_utc"])
      .groupby("carbon_timestamp_utc", as_index=False)[carbon_columns]
      .mean()
      .sort_values("carbon_timestamp_utc")
      .reset_index(drop=True)
)
carbon_long = carbon_wide.melt(
    id_vars=["carbon_timestamp_utc"],
    value_vars=carbon_columns,
    var_name="region",
    value_name="carbon_intensity"
)
carbon_long["region"] = carbon_long["region"].str.replace("carbon_intensity_", "", regex=False)
carbon_long = carbon_long.sort_values(["region", "carbon_timestamp_utc"]).reset_index(drop=True)

# Features use only prior observations for the same region.
for lag in [1, 2, 3, 6]:
    carbon_long[f"lag_{lag}"] = carbon_long.groupby("region")["carbon_intensity"].shift(lag)

carbon_long["rolling_mean_3"] = carbon_long.groupby("region")["carbon_intensity"].transform(
    lambda s: s.shift(1).rolling(3, min_periods=3).mean()
)
carbon_long["rolling_std_3"] = carbon_long.groupby("region")["carbon_intensity"].transform(
    lambda s: s.shift(1).rolling(3, min_periods=3).std()
)
carbon_long["hour"] = carbon_long["carbon_timestamp_utc"].dt.hour
carbon_long["day_of_week"] = carbon_long["carbon_timestamp_utc"].dt.dayofweek
carbon_long["month"] = carbon_long["carbon_timestamp_utc"].dt.month
carbon_long["target_next_carbon"] = carbon_long.groupby("region")["carbon_intensity"].shift(-1)
carbon_long["target_timestamp"] = carbon_long.groupby("region")["carbon_timestamp_utc"].shift(-1)

carbon_features_num = ["lag_1", "lag_2", "lag_3", "lag_6", "rolling_mean_3", "rolling_std_3",
                       "hour", "day_of_week", "month"]
carbon_features_cat = ["region"]
carbon_features = carbon_features_num + carbon_features_cat

carbon_model_df = carbon_long.dropna(subset=carbon_features_num + ["target_next_carbon", "target_timestamp"]).copy()
unique_target_times = np.sort(carbon_model_df["target_timestamp"].unique())
carbon_cutoff = unique_target_times[int(len(unique_target_times) * 0.80)]

train_carbon = carbon_model_df[carbon_model_df["target_timestamp"] <= carbon_cutoff].copy()
test_carbon = carbon_model_df[carbon_model_df["target_timestamp"] > carbon_cutoff].copy()

print("Regional carbon series shape:", carbon_long.shape)
print("Usable supervised rows:", len(carbon_model_df))
print("Train rows:", len(train_carbon), "| Test rows:", len(test_carbon))
print("Chronological cutoff:", carbon_cutoff)

In [ ]:
carbon_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), carbon_features_num),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), carbon_features_cat)
    ],
    remainder="drop"
)

carbon_model = Pipeline(steps=[
    ("preprocessor", carbon_preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=120,
        max_depth=18,
        min_samples_leaf=2,
        max_features=0.9,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

carbon_model.fit(train_carbon[carbon_features], train_carbon["target_next_carbon"])
pred_carbon = carbon_model.predict(test_carbon[carbon_features])

carbon_results = pd.DataFrame([
    regression_metrics(test_carbon["target_next_carbon"], pred_carbon, "Random Forest"),
    regression_metrics(
        test_carbon["target_next_carbon"],
        test_carbon["lag_1"],
        "Last-value baseline"
    )
])
display(carbon_results.round(3))

In [ ]:
carbon_predictions = test_carbon[["target_timestamp", "region", "target_next_carbon", "lag_1"]].copy()
carbon_predictions["predicted_next_carbon"] = pred_carbon
carbon_predictions["absolute_error"] = (
    carbon_predictions["target_next_carbon"] - carbon_predictions["predicted_next_carbon"]
).abs()
display(carbon_predictions.head(12))

fig, ax = plt.subplots(figsize=(12, 4))
sample = carbon_predictions.sort_values("target_timestamp").head(250)
ax.plot(sample["target_next_carbon"].to_numpy(), label="Actual", linewidth=1)
ax.plot(sample["predicted_next_carbon"].to_numpy(), label="Predicted", linewidth=1)
ax.set_title("Carbon intensity forecast (sample of chronological holdout)")
ax.set_ylabel("Dataset carbon-intensity units")
ax.legend()
plt.tight_layout()
plt.show()

## 4A. Compare additional algorithms

In addition to the Random Forest baseline, this section adds **Linear Regression**, **Gradient Boosting**, **XGBoost** (when installed), and **LSTM** for carbon time-series forecasting.

- **Linear Regression:** simple, interpretable baseline.
- **Gradient Boosting:** captures nonlinear feature interactions.
- **XGBoost:** strong gradient-boosted tree baseline; optional dependency.
- **LSTM:** sequence model for forecasting carbon intensity from previous time steps; TensorFlow is optional.

These are evaluated on the same chronological holdout. For a final research result, use a separate chronological validation split to select a model, then report the untouched test-set score. The existing scheduler remains wired to the Random Forest models until we deliberately select and validate an alternative.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

# Optional libraries: the notebook continues if they are not installed.
try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("XGBoost not installed. To include it, run: %pip install xgboost")

try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    HAS_TENSORFLOW = True
    tf.keras.utils.set_random_seed(RANDOM_STATE)
except ImportError:
    HAS_TENSORFLOW = False
    print("TensorFlow not installed. To include LSTM, run: %pip install tensorflow")

### 4A.1 Duration prediction: Linear Regression, Random Forest, Gradient Boosting and XGBoost

The same task features and chronological train/test split are used for each tabular algorithm. Categorical features are one-hot encoded and numeric features are median-imputed inside each pipeline.

In [ ]:
duration_algorithms = {
    "Linear Regression": Pipeline(steps=[
        ("preprocessor", duration_preprocessor),
        ("model", LinearRegression())
    ]),
    "Random Forest": duration_model,
    "Gradient Boosting": Pipeline(steps=[
        ("preprocessor", duration_preprocessor),
        ("model", GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.05, max_depth=3,
            random_state=RANDOM_STATE
        ))
    ])
}

if HAS_XGBOOST:
    duration_algorithms["XGBoost"] = Pipeline(steps=[
        ("preprocessor", duration_preprocessor),
        ("model", XGBRegressor(
            n_estimators=300, max_depth=5, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            objective="reg:squarederror", random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

duration_algorithm_rows = []
duration_algorithm_predictions = {}
for name, estimator in duration_algorithms.items():
    estimator.fit(X_train_duration, y_train_duration)
    pred = np.maximum(0, estimator.predict(X_test_duration))
    duration_algorithm_predictions[name] = pred
    duration_algorithm_rows.append(regression_metrics(y_test_duration, pred, name))

duration_algorithm_comparison = pd.DataFrame(duration_algorithm_rows).sort_values("MAE")
display(duration_algorithm_comparison.round(3))

### 4A.2 Carbon forecasting: compare tabular models

We compare the same four algorithms on lag-based carbon features. These features contain only past observations; the target is the next observed carbon-intensity value.

In [ ]:
carbon_algorithms = {
    "Linear Regression": Pipeline(steps=[
        ("preprocessor", carbon_preprocessor),
        ("model", LinearRegression())
    ]),
    "Random Forest": carbon_model,
    "Gradient Boosting": Pipeline(steps=[
        ("preprocessor", carbon_preprocessor),
        ("model", GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.05, max_depth=3,
            random_state=RANDOM_STATE
        ))
    ])
}

if HAS_XGBOOST:
    carbon_algorithms["XGBoost"] = Pipeline(steps=[
        ("preprocessor", carbon_preprocessor),
        ("model", XGBRegressor(
            n_estimators=300, max_depth=5, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            objective="reg:squarederror", random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

carbon_algorithm_rows = []
carbon_algorithm_predictions = {}
for name, estimator in carbon_algorithms.items():
    estimator.fit(train_carbon[carbon_features], train_carbon["target_next_carbon"])
    pred = estimator.predict(test_carbon[carbon_features])
    carbon_algorithm_predictions[name] = pred
    carbon_algorithm_rows.append(
        regression_metrics(test_carbon["target_next_carbon"], pred, name)
    )

carbon_algorithm_comparison = pd.DataFrame(carbon_algorithm_rows).sort_values("MAE")
display(carbon_algorithm_comparison.round(3))

### 4A.3 LSTM carbon time-series forecasting

LSTM is most naturally applied to the carbon time series, where the order of observations matters. Each input sequence contains the previous six observed values for one region; the target is the next value. The chronological split is by target timestamp.

This cell is optional because TensorFlow can be large to install. If TensorFlow is unavailable, the rest of the notebook still runs.

In [ ]:
lstm_carbon_metrics = None
lstm_carbon_model = None
lstm_carbon_predictions = None

if HAS_TENSORFLOW:
    LOOKBACK = 6
    carbon_times = carbon_wide["carbon_timestamp_utc"].to_numpy()
    carbon_values = carbon_wide[carbon_columns].to_numpy(dtype=np.float32)
    time_cut_index = int(len(carbon_wide) * 0.80)
    if time_cut_index <= LOOKBACK or len(carbon_wide) - time_cut_index < 2:
        print("Not enough timestamps for a reliable LSTM train/test split; skipping LSTM.")
    else:
        # Scale using training-period observations only to avoid test leakage.
        lstm_scaler = StandardScaler()
        lstm_scaler.fit(carbon_values[:time_cut_index].reshape(-1, 1))
        scaled_carbon = lstm_scaler.transform(carbon_values.reshape(-1, 1)).reshape(carbon_values.shape)

        X_seq, y_seq, y_time, y_region = [], [], [], []
        for region_idx, region_col in enumerate(carbon_columns):
            for target_idx in range(LOOKBACK, len(carbon_wide)):
                X_seq.append(scaled_carbon[target_idx-LOOKBACK:target_idx, region_idx])
                y_seq.append(scaled_carbon[target_idx, region_idx])
                y_time.append(carbon_times[target_idx])
                y_region.append(region_col.replace("carbon_intensity_", ""))

        X_seq = np.asarray(X_seq, dtype=np.float32)[..., np.newaxis]
        y_seq = np.asarray(y_seq, dtype=np.float32)
        y_time = np.asarray(y_time)
        train_mask = y_time <= carbon_times[time_cut_index - 1]
        test_mask = y_time > carbon_times[time_cut_index - 1]

        X_lstm_train, y_lstm_train = X_seq[train_mask], y_seq[train_mask]
        X_lstm_test, y_lstm_test = X_seq[test_mask], y_seq[test_mask]
        y_lstm_test_time = y_time[test_mask]
        y_lstm_test_region = np.asarray(y_region)[test_mask]

        lstm_carbon_model = Sequential([
            Input(shape=(LOOKBACK, 1)),
            LSTM(32, activation="tanh"),
            Dropout(0.1),
            Dense(16, activation="relu"),
            Dense(1)
        ])
        lstm_carbon_model.compile(optimizer="adam", loss="mse")
        early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
        lstm_carbon_model.fit(
            X_lstm_train, y_lstm_train,
            epochs=40, batch_size=32, validation_split=0.1,
            callbacks=[early_stop], verbose=0, shuffle=False
        )

        pred_scaled = lstm_carbon_model.predict(X_lstm_test, verbose=0).reshape(-1, 1)
        actual_scaled = y_lstm_test.reshape(-1, 1)
        pred_carbon_lstm = lstm_scaler.inverse_transform(pred_scaled).ravel()
        actual_carbon_lstm = lstm_scaler.inverse_transform(actual_scaled).ravel()
        lstm_carbon_predictions = pd.DataFrame({
            "target_timestamp": y_lstm_test_time,
            "region": y_lstm_test_region,
            "actual_carbon": actual_carbon_lstm,
            "predicted_carbon": pred_carbon_lstm
        })
        lstm_carbon_metrics = regression_metrics(
            actual_carbon_lstm, pred_carbon_lstm, "LSTM"
        )
        display(pd.DataFrame([lstm_carbon_metrics]).round(3))
else:
    print("LSTM skipped. Install TensorFlow and rerun this cell: %pip install tensorflow")

### 4A.4 Consolidated model comparison

This table collects the tabular-model scores and, when TensorFlow is available, the LSTM score. Compare both MAE and RMSE; lower is better. R² can be negative on difficult forecasting problems, so do not use it alone.

In [ ]:
carbon_all_rows = carbon_algorithm_rows.copy()
if lstm_carbon_metrics is not None:
    carbon_all_rows.append(lstm_carbon_metrics)

carbon_all_comparison = pd.DataFrame(carbon_all_rows).sort_values("MAE")
print("Duration prediction model comparison:")
display(duration_algorithm_comparison.round(3))
print("Carbon forecasting model comparison:")
display(carbon_all_comparison.round(3))

# Save comparison tables for the research report.
comparison_dir = Path("ml_artifacts")
comparison_dir.mkdir(exist_ok=True)
duration_algorithm_comparison.to_csv(comparison_dir / "duration_algorithm_comparison.csv", index=False)
carbon_all_comparison.to_csv(comparison_dir / "carbon_algorithm_comparison.csv", index=False)

## 5. Build a one-step-ahead carbon forecast lookup

For a given region, we use the latest observed lag values to predict its next carbon-intensity value. If a region has too little history or the model cannot generate a forecast, we fall back to its latest observation.

This function is for a reproducible demonstration; a real simulation should update the history as simulated time advances.

In [ ]:
def forecast_next_carbon_for_region(region_name, carbon_long_df=carbon_long, model=carbon_model):
    region_name = str(region_name).lower().replace(" ", "_")
    hist = carbon_long_df[carbon_long_df["region"] == region_name].sort_values("carbon_timestamp_utc")
    if hist.empty:
        return np.nan

    latest = hist.iloc[-1]
    # Need the same lag logic as training. Use the original carbon series and recompute features.
    vals = hist["carbon_intensity"].dropna().to_numpy()
    if len(vals) < 7:
        return float(vals[-1])

    ts = hist["carbon_timestamp_utc"].iloc[-1] + (hist["carbon_timestamp_utc"].diff().dropna().median())
    if pd.isna(ts):
        ts = hist["carbon_timestamp_utc"].iloc[-1]

    row = {
        "region": region_name,
        "lag_1": vals[-1],
        "lag_2": vals[-2],
        "lag_3": vals[-3],
        "lag_6": vals[-6],
        "rolling_mean_3": np.mean(vals[-3:]),
        "rolling_std_3": np.std(vals[-3:], ddof=1),
        "hour": ts.hour,
        "day_of_week": ts.dayofweek,
        "month": ts.month
    }
    X_next = pd.DataFrame([row])[carbon_features]
    return float(max(0, model.predict(X_next)[0]))

latest_carbon_by_region = (
    carbon_long.sort_values("carbon_timestamp_utc")
    .groupby("region", as_index=False)
    .tail(1)
    .set_index("region")["carbon_intensity"]
    .to_dict()
)

carbon_forecast_lookup = {}
for region_name in sorted(carbon_long["region"].unique()):
    carbon_forecast_lookup[region_name] = forecast_next_carbon_for_region(region_name)

display(pd.DataFrame({
    "region": list(carbon_forecast_lookup.keys()),
    "latest_observed_carbon": [latest_carbon_by_region.get(r, np.nan) for r in carbon_forecast_lookup],
    "forecast_next_carbon": list(carbon_forecast_lookup.values())
}).sort_values("forecast_next_carbon"))

## 6. Carbon-aware edge/cloud scheduler prototype

The scheduler scores feasible placements using:
- predicted task duration,
- estimated energy,
- estimated carbon emissions,
- deadline feasibility,
- GPU feasibility.

### Explicit assumptions (replace with measured/simulator values later)
1. `cpu_demand` is treated as a relative workload indicator, not a measured utilization percentage.
2. Approximate utilization proxy = `cpu_demand / max(cpu_demand in training data)`, clipped to 0–1.
3. Power is linearly interpolated between idle and full-load power:  
   `P(u) = P_idle + (P_full - P_idle) × u`
4. Energy in kWh = `power_W × duration_sec / 3,600,000`.
5. CO₂ in grams = `energy_kWh × carbon_intensity_g_per_kWh`.
6. Because measured network latency is not supplied, this prototype uses task duration as its execution-time estimate and does not claim to model network latency accurately.
7. The workbook states zero edge GPU cards and eight cloud GPU cards. Tasks with positive GPU demand are therefore treated as cloud-only in this prototype.
8. Carbon-aware scoring weights below are tunable research parameters, not learned parameters.

In [ ]:
# These constants come from the workbook's infrastructure columns.
example = df.iloc[0]
EDGE_IDLE_W = float(df["edge_idle_power_watts"].median())
EDGE_FULL_W = float(df["edge_full_load_power_watts"].median())
CLOUD_IDLE_W = float(df["cloud_idle_power_watts"].median())
CLOUD_FULL_W = float(df["cloud_full_load_power_watts"].median())

EDGE_GPU_CARDS = int(df["edge_cap_gpu_cards"].median())
CLOUD_GPU_CARDS = int(df["cloud_cap_gpu_cards"].median())
CPU_DEMAND_SCALE = max(float(train_duration["cpu_demand"].max()), 1.0)

# Scheduler weights: increase carbon weight to favor lower emissions;
# increase deadline weight to penalize placements likely to miss deadlines.
WEIGHT_CARBON = 0.55
WEIGHT_LATENCY = 0.25
WEIGHT_ENERGY = 0.20
DEADLINE_PENALTY = 10.0

print({
    "edge_idle_w": EDGE_IDLE_W, "edge_full_w": EDGE_FULL_W,
    "cloud_idle_w": CLOUD_IDLE_W, "cloud_full_w": CLOUD_FULL_W,
    "edge_gpu_cards": EDGE_GPU_CARDS, "cloud_gpu_cards": CLOUD_GPU_CARDS
})

In [ ]:
def estimate_power_watts(cpu_demand, idle_w, full_w):
    utilization_proxy = np.clip(float(cpu_demand) / CPU_DEMAND_SCALE, 0.0, 1.0)
    return float(idle_w + (full_w - idle_w) * utilization_proxy)

def estimate_energy_kwh(power_w, duration_sec):
    return float(power_w * max(float(duration_sec), 0.0) / 3_600_000.0)

def score_candidate(duration_sec, power_w, carbon_intensity, deadline_sec):
    energy_kwh = estimate_energy_kwh(power_w, duration_sec)
    carbon_g = energy_kwh * float(carbon_intensity)

    # Normalize to make the weighted score less dominated by raw units.
    # These reference scales are transparent tuning constants.
    latency_norm = max(float(duration_sec), 0.0) / max(float(deadline_sec), 1.0)
    carbon_norm = carbon_g / 100.0
    energy_norm = energy_kwh / 0.01

    deadline_miss = float(duration_sec > deadline_sec)
    score = (
        WEIGHT_CARBON * carbon_norm
        + WEIGHT_LATENCY * latency_norm
        + WEIGHT_ENERGY * energy_norm
        + DEADLINE_PENALTY * deadline_miss
    )
    return {
        "duration_sec": float(duration_sec),
        "power_w": float(power_w),
        "energy_kwh": energy_kwh,
        "estimated_co2_g": carbon_g,
        "deadline_miss": bool(deadline_miss),
        "score": float(score)
    }

def choose_placement(task_row, predicted_duration_sec, carbon_local, carbon_cloud):
    deadline = float(task_row.get("delay_tolerance_sec", np.inf))
    cpu = float(task_row.get("cpu_demand", 0))
    gpu = float(task_row.get("gpu_demand", 0))

    candidates = {}
    if gpu <= 0 or EDGE_GPU_CARDS > 0:
        edge_power = estimate_power_watts(cpu, EDGE_IDLE_W, EDGE_FULL_W)
        candidates["edge"] = score_candidate(
            predicted_duration_sec, edge_power, carbon_local, deadline
        )

    if gpu <= 0 or CLOUD_GPU_CARDS > 0:
        cloud_power = estimate_power_watts(cpu, CLOUD_IDLE_W, CLOUD_FULL_W)
        # Same predicted workload duration as a base estimate. A real simulator should
        # replace this with placement-specific execution-time estimates.
        candidates["cloud"] = score_candidate(
            predicted_duration_sec, cloud_power, carbon_cloud, deadline
        )

    if not candidates:
        return {"placement": "no_feasible_candidate", "candidates": candidates}

    best = min(candidates, key=lambda name: candidates[name]["score"])
    return {"placement": best, "candidates": candidates}

## 7. Run scheduler on a held-out sample

We use duration predictions from the chronological holdout. Carbon intensity for the task's local region is forecast using that region's series; cloud carbon is forecast for California, which is the cloud region identified in this dataset. This example uses a static latest-history forecast lookup, so treat it as an integration demonstration rather than a time-evolving simulation result.

In [ ]:
# Build a held-out sample with predictions.
scheduler_test = test_duration.copy()
scheduler_test["predicted_duration_sec"] = pred_duration

def normalize_region_name(value):
    return str(value).strip().lower().replace(" ", "_")

scheduler_rows = []
for _, row in scheduler_test.head(500).iterrows():
    local_region = normalize_region_name(row.get("device_region", ""))
    # The workbook's cloud_region is California in the current dataset.
    cloud_region = normalize_region_name(row.get("cloud_region", "california"))

    local_carbon = carbon_forecast_lookup.get(local_region, row.get("carbon_intensity_local", np.nan))
    cloud_carbon = carbon_forecast_lookup.get(cloud_region, row.get("carbon_intensity_cloud", np.nan))

    if pd.isna(local_carbon):
        local_carbon = float(row.get("carbon_intensity_local", 350.0))
    if pd.isna(cloud_carbon):
        cloud_carbon = float(row.get("carbon_intensity_cloud", 350.0))

    decision = choose_placement(
        row,
        float(row["predicted_duration_sec"]),
        float(local_carbon),
        float(cloud_carbon)
    )

    chosen = decision["placement"]
    chosen_metrics = decision["candidates"].get(chosen, {})
    scheduler_rows.append({
        "task_id": row.get("task_id"),
        "priority": row.get("priority"),
        "gpu_demand": row.get("gpu_demand"),
        "predicted_duration_sec": row["predicted_duration_sec"],
        "deadline_sec": row.get("delay_tolerance_sec"),
        "local_carbon_forecast": local_carbon,
        "cloud_carbon_forecast": cloud_carbon,
        "placement": chosen,
        "estimated_energy_kwh": chosen_metrics.get("energy_kwh", np.nan),
        "estimated_co2_g": chosen_metrics.get("estimated_co2_g", np.nan),
        "deadline_miss_predicted": chosen_metrics.get("deadline_miss", np.nan),
        "scheduler_score": chosen_metrics.get("score", np.nan)
    })

scheduler_results = pd.DataFrame(scheduler_rows)
display(scheduler_results.head(15))

In [ ]:
print("Tasks scheduled:", len(scheduler_results))
print("\nPlacement counts:")
display(scheduler_results["placement"].value_counts().to_frame("tasks"))

print("\nEstimated aggregate metrics (model-based estimates, not measured energy):")
display(pd.DataFrame([{
    "tasks": len(scheduler_results),
    "estimated_total_energy_kwh": scheduler_results["estimated_energy_kwh"].sum(),
    "estimated_total_co2_g": scheduler_results["estimated_co2_g"].sum(),
    "predicted_deadline_miss_rate": scheduler_results["deadline_miss_predicted"].astype(float).mean()
}]).round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
scheduler_results["placement"].value_counts().plot(kind="bar", ax=axes[0])
axes[0].set_title("Placement decisions")
axes[0].set_ylabel("Tasks")
scheduler_results.groupby("placement")["estimated_co2_g"].mean().plot(kind="bar", ax=axes[1])
axes[1].set_title("Mean estimated CO₂ by placement")
axes[1].set_ylabel("Estimated grams per task")
plt.tight_layout()
plt.show()

## 8. Simple baseline comparison

This section compares the proposed scheduler's selected placement against two simple policies on the same held-out sample:
- **Always edge when feasible:** otherwise cloud.
- **Lowest current carbon:** choose the feasible candidate with lower carbon intensity, ignoring duration/energy weights.

This is an initial sanity-check comparison. For a research paper, implement the full baselines previously discussed: Round Robin, reactive threshold, CPU-aware, latency-aware, and carbon-aware without forecasting. Compare all policies on identical workloads and simulation seeds.

In [ ]:
def evaluate_policy_for_task(row, predicted_duration, local_carbon, cloud_carbon, policy):
    deadline = float(row.get("delay_tolerance_sec", np.inf))
    cpu = float(row.get("cpu_demand", 0))
    gpu = float(row.get("gpu_demand", 0))
    feasible = []
    if gpu <= 0 or EDGE_GPU_CARDS > 0:
        feasible.append(("edge", estimate_power_watts(cpu, EDGE_IDLE_W, EDGE_FULL_W), local_carbon))
    if gpu <= 0 or CLOUD_GPU_CARDS > 0:
        feasible.append(("cloud", estimate_power_watts(cpu, CLOUD_IDLE_W, CLOUD_FULL_W), cloud_carbon))
    if not feasible:
        return None

    scored = {
        name: score_candidate(predicted_duration, power, carbon, deadline)
        for name, power, carbon in feasible
    }
    if policy == "proposed_weighted":
        chosen = min(scored, key=lambda name: scored[name]["score"])
    elif policy == "edge_first":
        chosen = "edge" if "edge" in scored else "cloud"
    elif policy == "lowest_carbon":
        chosen = min(feasible, key=lambda item: item[2])[0]
    else:
        raise ValueError("Unknown policy")

    return {"placement": chosen, **scored[chosen]}

comparison_records = []
for _, row in scheduler_test.head(500).iterrows():
    local_region = normalize_region_name(row.get("device_region", ""))
    cloud_region = normalize_region_name(row.get("cloud_region", "california"))
    local_carbon = carbon_forecast_lookup.get(local_region, float(row.get("carbon_intensity_local", 350.0)))
    cloud_carbon = carbon_forecast_lookup.get(cloud_region, float(row.get("carbon_intensity_cloud", 350.0)))
    local_carbon = float(local_carbon) if pd.notna(local_carbon) else float(row.get("carbon_intensity_local", 350.0))
    cloud_carbon = float(cloud_carbon) if pd.notna(cloud_carbon) else float(row.get("carbon_intensity_cloud", 350.0))

    for policy in ["proposed_weighted", "edge_first", "lowest_carbon"]:
        result = evaluate_policy_for_task(row, float(row["predicted_duration_sec"]), local_carbon, cloud_carbon, policy)
        if result:
            comparison_records.append({
                "policy": policy,
                "placement": result["placement"],
                "energy_kwh": result["energy_kwh"],
                "estimated_co2_g": result["estimated_co2_g"],
                "deadline_miss": result["deadline_miss"]
            })

comparison_df = pd.DataFrame(comparison_records)
comparison_summary = comparison_df.groupby("policy").agg(
    tasks=("placement", "size"),
    total_estimated_energy_kwh=("energy_kwh", "sum"),
    total_estimated_co2_g=("estimated_co2_g", "sum"),
    predicted_deadline_miss_rate=("deadline_miss", "mean")
).reset_index()
display(comparison_summary.round(4))

## 9. Save models and outputs

These artifacts can be used later by a Python inference service or exported into a Java/CloudSim integration workflow. The Java simulator should send task features to Python (for example through a REST API or a CSV handoff) and receive predicted duration/carbon values and a placement recommendation.

In [ ]:
ARTIFACT_DIR = Path("ml_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

joblib.dump(duration_model, ARTIFACT_DIR / "duration_random_forest.joblib")
joblib.dump(carbon_model, ARTIFACT_DIR / "carbon_forecast_random_forest.joblib")

duration_predictions.to_csv(ARTIFACT_DIR / "duration_test_predictions.csv", index=False)
carbon_predictions.to_csv(ARTIFACT_DIR / "carbon_test_predictions.csv", index=False)
scheduler_results.to_csv(ARTIFACT_DIR / "scheduler_demo_results.csv", index=False)
comparison_summary.to_csv(ARTIFACT_DIR / "scheduler_baseline_comparison.csv", index=False)

print("Saved artifacts to:", ARTIFACT_DIR.resolve())
print(sorted(p.name for p in ARTIFACT_DIR.iterdir()))

## 10. How to run a prediction for a new task

The task should contain the same feature columns used for training. This example uses a row from the dataset; replace it with a new task dictionary when integrating the scheduler.

In [ ]:
example_task = df.iloc[[0]].copy()
example_duration = max(0.0, float(duration_model.predict(example_task[duration_features])[0]))

example_local_region = normalize_region_name(example_task.iloc[0].get("device_region", ""))
example_cloud_region = normalize_region_name(example_task.iloc[0].get("cloud_region", "california"))
example_local_carbon = carbon_forecast_lookup.get(
    example_local_region, float(example_task.iloc[0].get("carbon_intensity_local", 350.0))
)
example_cloud_carbon = carbon_forecast_lookup.get(
    example_cloud_region, float(example_task.iloc[0].get("carbon_intensity_cloud", 350.0))
)

example_decision = choose_placement(
    example_task.iloc[0],
    example_duration,
    float(example_local_carbon),
    float(example_cloud_carbon)
)

print("Predicted duration (sec):", round(example_duration, 2))
print("Forecast local carbon:", round(float(example_local_carbon), 2))
print("Forecast cloud carbon:", round(float(example_cloud_carbon), 2))
print("Placement:", example_decision["placement"])
display(pd.DataFrame(example_decision["candidates"]).T)

## 11. Interpretation, limitations, and next research steps

### What this implementation contributes
- A reproducible ML pipeline using the supplied dataset.
- Duration prediction using task metadata.
- One-step-ahead regional carbon forecasting using past-only lag features.
- A forecast-aware, multi-objective placement prototype.
- Initial baselines and saved outputs.

### Limitations to document
- `observed_duration_sec` may reflect the source trace's execution environment; it is not guaranteed to equal execution time in our simulated edge/cloud infrastructure.
- Carbon values are forecast at the available observation interval. Longer horizons need separate evaluation.
- Energy is estimated from the supplied idle/full power values and a CPU-demand proxy; it is not a hardware measurement.
- Network latency and queue length are not directly measured in this workbook, so this prototype does not claim to optimize them precisely.
- The scheduler's weights and normalizers are hand-set parameters and should be tuned via sensitivity analysis.
- Random Forest is our first baseline. After validating this pipeline, compare XGBoost/HistGradientBoosting and a temporal model (for example LSTM) if enough time-series history is available.

### Recommended next experiment
Integrate the prediction function with CloudSim/EdgeCloudSim and compare:
1. Round Robin
2. Reactive CPU-threshold scheduling
3. CPU-aware scheduling
4. Carbon-aware scheduling without forecasting
5. **Forecast-driven carbon-aware scheduling (our proposed method)**

Use the same task trace and random seeds for every policy, then report mean latency, deadline violation rate, energy, CO₂ emissions, and utilization/overload statistics.